# Model input EDA: the rows the statistical models are fit on

`analysis_eda.ipynb` explores all of `analysis_measurement`. The models don't see all of it. They keep
treated and control rows only, drop flagged rows (non-perennial reaches, upstream stations in the old
reservoir, SSC/TSS without same-day flow, the suspect dams), collapse same-day repeats to one daily
median, need both periods per dam-station pair, and drop rows missing a modifier or a covariate.
This notebook rebuilds **exactly those rows**, with the models' own code
(`mixed_model.prepare` / `model_rows`, `distance_models.restrict` / `model_frame`), and explores them.
It fits nothing.

| Run | Code | Treated rows | Specs |
|---|---|---|---|
| `mixed_10km` | `analysis/mixed_model.py` | downstream ≤ 10 km | `before_after`, `baci` |
| `distance_30km` | `analysis/distance_models.py` (Stage 3–4 primary) | downstream ≤ 30 km, suspect dams out | distance, scaled distance, size |

**Sections:** 1 where the rows go · 2 sample size per model · 3 map of the model dams ·
4 how concentrated the data is · 5 pre/post balance · 6 outcome distributions and outliers ·
7 a model-free BACI check · 8 distance and dilution · 9 reservoir size ·
10 dam, watershed and climate characteristics · 11 covariate shifts · 12 checklist.

Run on the cluster after `build_analysis_dataset` (a few minutes; set `DAM_REMOVAL_ANALYSIS_INPUT`
to read another `analysis_measurement` .parquet or .csv).

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "etl").is_dir() and (p / "config").is_dir())
sys.path.insert(0, str(ROOT))

from analysis import distance_models as dm
from analysis import mixed_model as mm
from etl.common import wq_rules

CASE_STUDY = "all_removals"
INPUT = os.environ.get("DAM_REMOVAL_ANALYSIS_INPUT")
if INPUT is None:
    from etl.common.config import load_config
    from etl.common.warehouse import warehouse_dir
    INPUT = warehouse_dir(load_config(CASE_STUDY, project_root=ROOT)) / "analysis_measurement.parquet"
INPUT = Path(INPUT)

# The two runs, with the settings the cluster jobs use (mixed_model.SETTINGS;
# distance_models.load_daily with its defaults).
RUNS = {
    "mixed_10km": {"settings": mm.SETTINGS, "specs": list(mm.MODELS), "exclude": []},
    "distance_30km": {
        "settings": {**mm.SETTINGS, "max_downstream_km": 30.0, "carry_columns": dm.CARRY,
                     "min_dams": 10, "exploratory_below": 15, "series": dm.default_series()},
        "specs": dm.DEFAULT_SPECS + ["baci_size", "before_after_size"],
        "exclude": dm.read_exclusions(ROOT / "config" / "suspect_dam_matches.csv"),
    },
}
DM = RUNS["distance_30km"]["settings"]
BAND_EDGES = [0, 1, 5, 10, 20, 30]

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format",   # years and counts in full, the rest to 3 significant digits
              lambda v: f"{v:.0f}" if 1000 <= abs(v) < 1e7 else f"{v:,.3g}")

df = pd.read_parquet(INPUT) if INPUT.suffix == ".parquet" else pd.read_csv(INPUT, low_memory=False)
df["series"] = mm.series_name(df)
print(f"{INPUT}\n{len(df):,} rows, {df['AR_ID'].nunique()} dams, {df['station_id'].nunique():,} stations")
print("Suspect dams left out of the 30 km run:", len(RUNS["distance_30km"]["exclude"]))


def y_label(series):
    parameter = series.split(" (")[0]
    unit = wq_rules.STANDARD_UNITS.get(parameter, "")
    return f"log10 {unit}" if parameter in DM["log_parameters"] else unit


def facet_axes(n, ncols=4, width=4, height=3):
    nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(width * ncols, height * nrows), squeeze=False)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, list(axes.flat)

## 1. Where the rows of `analysis_measurement` go

Every row gets the **first** reason it is left out, in the order the model code applies them.
The last cell checks that the rows marked `kept` are exactly the rows `mixed_model.prepare` keeps,
so this table can't drift from the code.

- `series_not_modeled`: a parameter / fraction the models don't fit (e.g. nitrate, dissolved phosphorus).
- `downstream_not_assigned`: downstream of this dam, but the sample is credited to another removal,
  or two removals tie, or the distance is unknown.
- `control_exposed`: upstream of this dam but downstream of another removal in its window.
- `in_former_reservoir`: upstream station probably inside the old reservoir (affected, not a control).
- `side_unknown`: a same-reach station exactly at the dam.
- `beyond_km_limit`: treated row farther than the run's limit, or with no distance.
- `non_perennial`, `no_same_day_discharge`: the flags (SSC/TSS need same-day flow).
- `no_value`: no standardized value.

In [ ]:
def exclusion_reason(frame, settings):
    """First reason each row is left out (mixed_model.assign_roles + prepare order), else 'kept'."""
    side, status = frame["side"], frame["exposure_status"]
    km = pd.to_numeric(frame["network_distance_km"], errors="coerce")
    in_res = frame["upstream_in_former_reservoir"].fillna(False).astype(bool)
    flagged = {flag: frame[flag].fillna(False).astype(bool) for flag in settings["exclude_flags"]}
    no_flow = (frame["parameter_std"].isin(settings["require_same_day_discharge"])
               & ~frame["has_same_day_discharge"].fillna(False).astype(bool))
    limit = settings.get("max_downstream_km")
    conditions = [
        ("series_not_modeled", ~frame["series"].isin(settings["series"])),
        ("side_unknown", side.isna() | ~side.isin(["downstream", "upstream"])),
        ("downstream_not_assigned", side.eq("downstream") & ~status.isin(settings["treated_exposure"])),
        ("control_exposed", side.eq("upstream") & ~status.isin(settings["control_exposure"])),
        ("in_former_reservoir", side.eq("upstream") & in_res & settings["exclude_former_reservoir"]),
        ("beyond_km_limit", side.eq("downstream") & (~km.le(limit).fillna(False) if limit else False)),
        *[(flag.removeprefix("qc_"), mask) for flag, mask in flagged.items()],
        ("no_same_day_discharge", no_flow),
        ("no_value", frame["result_value_std"].isna()),
    ]
    reason = pd.Series("kept", index=frame.index)
    for name, mask in reversed(conditions):      # the first condition that holds wins
        reason = reason.mask(mask, name)
    return reason


funnel = {}
for run, spec in RUNS.items():
    s = spec["settings"]
    reason = exclusion_reason(df, s)
    role = mm.assign_roles(df, s)
    expected = df["series"].isin(s["series"]) & role.ne("excluded") & df["result_value_std"].notna()
    assert reason.eq("kept").equals(expected), f"{run}: reasons disagree with mixed_model.assign_roles"
    df[f"reason_{run}"], df[f"role_{run}"] = reason, role
    funnel[run] = reason.value_counts()

funnel = pd.DataFrame(funnel).fillna(0).astype(int)
funnel["share_30km_%"] = (100 * funnel["distance_30km"] / len(df)).round(1)
display(funnel.sort_values("distance_30km", ascending=False))

In [ ]:
# Kept rows by series, and how far the daily collapse shrinks them (30 km run).
kept = df[df["reason_distance_30km"].eq("kept")]
by_series = kept.groupby("series").agg(
    rows=("measurement_id", "size"), dams=("AR_ID", "nunique"), stations=("station_id", "nunique"))
by_series["days"] = (kept.drop_duplicates(["AR_ID", "station_id", "series", "sample_date"])
                     .groupby("series").size())
by_series["rows_per_day"] = by_series["rows"] / by_series["days"]
display(by_series.sort_values("dams", ascending=False))
# rows_per_day well above 1 = continuous loggers or depth profiles; the median collapse
# keeps them from counting as extra evidence.

## 2. Sample size of every model

Rebuilt with the models' own functions. `dams_before_modifier` → `dams`: dams lost because a modifier
(distance, drainage-area ratio, backwater length, size) is missing. `fit` says whether the model runs
(`min_dams`) and whether it is labelled exploratory (fewer than 15 dams in the 30 km run).

In [ ]:
DAILY, FRAMES, sizes = {}, {}, []
for run, spec in RUNS.items():
    s = spec["settings"]
    daily = mm.prepare(df, s)
    restricted, counts = dm.restrict(daily, spec["exclude"])
    DAILY[run] = restricted
    for series in s["series"]:
        for model in spec["specs"]:
            if run == "mixed_10km":
                rows = mm.model_rows(restricted[restricted["series"] == series], model, s)
                frame = rows.dropna(subset=s["covariates"])
                meta = {"dams_before_modifier": rows["AR_ID"].nunique(),
                        "rows_dropped_missing_covariates": len(rows) - len(frame)}
            else:
                frame, _, meta = dm.model_frame(restricted, series, model, s)
            FRAMES[(run, series, model)] = frame
            role_days = frame["role"].value_counts()
            dams = frame["AR_ID"].nunique()
            fit = ("skipped" if dams < s["min_dams"]
                   else "exploratory" if dams < s.get("exploratory_below", 0) else "fit")
            sizes.append({"run": run, "series": series, "model": model, "dams": dams,
                          "control_dams": frame.loc[frame["role"].eq("control"), "AR_ID"].nunique(),
                          "treated_stations": frame.loc[frame["role"].eq("treated"), "station_id"].nunique(),
                          "control_stations": frame.loc[frame["role"].eq("control"), "station_id"].nunique(),
                          "days": len(frame), "treated_days": role_days.get("treated", 0),
                          "control_days": role_days.get("control", 0),
                          "post_share": frame["post"].mean() if len(frame) else np.nan,
                          **meta, "fit": fit})

sizes = pd.DataFrame(sizes)
display(sizes.pivot_table(index="series", columns=["run", "model"], values="dams", aggfunc="first")
        .astype("Int64"))

In [ ]:
# The full table for the 30 km primary specs.
primary = sizes[(sizes["run"] == "distance_30km")
                & sizes["model"].isin(["baci_distance", "before_after_distance"])]
display(primary.drop(columns="run").sort_values(["model", "dams"], ascending=[True, False]))

## 3. Where the model dams are

One dot per dam in the chosen run's models. **Red**: the dam has upstream controls (in a BACI model);
**blue**: downstream only (before/after models only). Dot size grows with the number of measures the dam
is used for. Each measure has its own layer (off by default; turn it on in the layer box). Click a dot
for the dam's measures, its downstream / upstream stations and days. Set `MAP_RUN` to `"mixed_10km"`
for the 10 km models. Coordinates are the reported ones from the warehouse `dam_removal` table
(`DAM_REMOVAL_WAREHOUSE_DIR` reads another folder).

In [ ]:
import folium

MAP_RUN = "distance_30km"
DESIGN_SPECS = {"distance_30km": {"baci": "baci_distance", "before_after": "before_after_distance"},
                "mixed_10km": {"baci": "baci", "before_after": "before_after"}}[MAP_RUN]

# Per dam, measure and design: its stations and days in that model.
used = []
for series in RUNS[MAP_RUN]["settings"]["series"]:
    for design, model in DESIGN_SPECS.items():
        frame = FRAMES[(MAP_RUN, series, model)]
        if frame["AR_ID"].nunique() < RUNS[MAP_RUN]["settings"]["min_dams"]:
            continue   # the model isn't fit, so its dams aren't "used"
        used.append(frame.groupby("AR_ID").agg(
            treated_stations=("station_id", lambda s: s[frame.loc[s.index, "role"].eq("treated")].nunique()),
            control_stations=("station_id", lambda s: s[frame.loc[s.index, "role"].eq("control")].nunique()),
            days=("y", "size")).reset_index().assign(series=series, design=design))
used = pd.concat(used, ignore_index=True) if used else pd.DataFrame(columns=["AR_ID", "series", "design"])
used["AR_ID"] = used["AR_ID"].astype(str)

WAREHOUSE = os.environ.get("DAM_REMOVAL_WAREHOUSE_DIR")
if WAREHOUSE is None:
    from etl.common.config import load_config
    from etl.common.warehouse import warehouse_dir
    WAREHOUSE = warehouse_dir(load_config(CASE_STUDY, project_root=ROOT))
dams = pd.read_parquet(Path(WAREHOUSE) / "dam_removal.parquet",
                       columns=["AR_ID", "Dam_Name", "River", "State", "Year_Removed", "Latitude", "Longitude"])
dams = dams.assign(AR_ID=dams["AR_ID"].astype(str)).set_index("AR_ID")

per_dam = (used.groupby("AR_ID").agg(n_measures=("series", "nunique"),
                                     baci=("design", lambda d: d.eq("baci").any()))
           .join(dams, how="left"))
missing = per_dam["Latitude"].isna() | per_dam["Longitude"].isna()
if missing.any():
    print(f"{int(missing.sum())} model dams have no coordinates in dam_removal: "
          f"{sorted(per_dam.index[missing])[:10]}")
per_dam = per_dam[~missing]
RED, BLUE = "#c0392b", "#2980b9"


def popup(aid, row, series=None):
    rows = used[used["AR_ID"].eq(aid) & (used["series"].eq(series) if series else True)]
    table = "".join(f"<tr><td>{s}</td><td>{'BACI' if d == 'baci' else 'before/after'}</td>"
                    f"<td>{t} down, {c} up</td><td>{n:,} days</td></tr>"
                    for s, d, t, c, n in rows[["series", "design", "treated_stations",
                                               "control_stations", "days"]].itertuples(index=False))
    return folium.Popup(f"<b>{row.Dam_Name}</b> ({aid})<br>{row.River}, {row.State}; removed "
                        f"{row.Year_Removed}<table>{table}</table>", max_width=420)


# Carto light basemap by URL (folium 0.20 asks for an API key for its "CartoDB positron" name).
m = folium.Map(location=[39.5, -96], zoom_start=4,
               tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
               attr="&copy; OpenStreetMap contributors &copy; CARTO")
everything = folium.FeatureGroup(name=f"All model dams ({len(per_dam)}, {int(per_dam['baci'].sum())} BACI)",
                                 show=True)
for aid, row in per_dam.iterrows():
    folium.CircleMarker([row.Latitude, row.Longitude], radius=3 + 0.7 * row.n_measures,
                        color=RED if row.baci else BLUE, fill=True, fill_opacity=0.7, weight=1,
                        popup=popup(aid, row), tooltip=f"{row.Dam_Name}: {row.n_measures} measures"
                        ).add_to(everything)
everything.add_to(m)
for series, g in used.groupby("series"):
    baci = g.groupby("AR_ID")["design"].apply(lambda d: d.eq("baci").any())
    layer = folium.FeatureGroup(name=f"{series} ({len(baci)} dams, {int(baci.sum())} BACI)", show=False)
    for aid, is_baci in baci.items():
        if aid in per_dam.index:
            row = per_dam.loc[aid]
            folium.CircleMarker([row.Latitude, row.Longitude], radius=5, color=RED if is_baci else BLUE,
                                fill=True, fill_opacity=0.8, weight=1, popup=popup(aid, row, series),
                                tooltip=row.Dam_Name).add_to(layer)
    layer.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.get_root().html.add_child(folium.Element(
    f'<div style="position:fixed;bottom:25px;left:25px;z-index:9999;background:white;padding:6px 10px;'
    f'border:1px solid #999;font-size:12px"><b>{MAP_RUN}</b><br>'
    f'<span style="color:{RED}">&#9679;</span> has upstream controls (BACI)'
    f'<br><span style="color:{BLUE}">&#9679;</span> downstream only (before/after)</div>'))
display(m)

# Dams per state, for the same run.
display(per_dam.assign(State=per_dam["State"].fillna("?"))
        .groupby("State").agg(dams=("baci", "size"), baci_dams=("baci", "sum"))
        .sort_values("dams", ascending=False).T)

## 4. How concentrated the evidence is

Dam random effects make each dam one unit of evidence for the effect, but a dam with thousands of
logger days still sets its own effect precisely. If one or two dams hold most of the days, check the
leave-one-dam-out results for that series.

In [ ]:
def concentration(frame):
    days = frame.groupby("AR_ID").size().sort_values(ascending=False)
    total = days.sum()
    return pd.Series({"dams": len(days), "median_days_per_dam": days.median(),
                      "top_dam": days.index[0] if len(days) else None,
                      "top_dam_share_%": 100 * days.iloc[0] / total if total else np.nan,
                      "top5_share_%": 100 * days.iloc[:5].sum() / total if total else np.nan})


conc = pd.DataFrame({key: concentration(f) for key, f in FRAMES.items()
                     if key[0] == "distance_30km" and key[2] in ("baci_distance", "before_after_distance")
                     and len(f)}).T
conc.index.names = ["run", "series", "model"]
display(conc.droplevel("run").sort_values("top_dam_share_%", ascending=False))

In [ ]:
# Days per dam, before/after model (30 km): a long right tail = a few heavily sampled dams.
fig, axes = facet_axes(len(DM["series"]))
for ax, series in zip(axes, DM["series"]):
    frame = FRAMES[("distance_30km", series, "before_after_distance")]
    if frame.empty:
        ax.set_title(f"{series}: no rows"); continue
    ax.hist(np.log10(frame.groupby("AR_ID").size()), bins=20, color="grey")
    ax.set(title=series, xlabel="log10 days per dam", ylabel="dams")
fig.tight_layout()

## 5. Before vs after balance

The month fixed effects and climate covariates adjust for season and weather, but only if both periods
cover the same months. Large shifts in sampling month, provider, censoring or turbidity unit between
periods can look like a removal effect. Each comparison uses the **30 km BACI** rows unless noted.

**Total variation distance (TVD)** below = half the summed absolute difference between two distributions:
0 identical, 1 no overlap. Above ~0.2 is worth a look.

In [ ]:
def tvd(a, b):
    p, q = a / a.sum(), b / b.sum()
    p, q = p.align(q, fill_value=0)
    return 0.5 * (p - q).abs().sum()


rows = []
for series in DM["series"]:
    frame = FRAMES[("distance_30km", series, "baci_distance")]
    for role in ("treated", "control"):
        part = frame[frame["role"] == role]
        if part.empty:
            continue
        months = part.groupby(["post", "month"]).size().unstack(0, fill_value=0)
        if months.shape[1] < 2:
            continue
        rows.append({"series": series, "role": role, "pre_days": int(months[0].sum()),
                     "post_days": int(months[1].sum()), "month_tvd": tvd(months[0], months[1]),
                     "summer_share_pre": months.loc[months.index.isin([6, 7, 8]), 0].sum() / months[0].sum(),
                     "summer_share_post": months.loc[months.index.isin([6, 7, 8]), 1].sum() / months[1].sum()})
month_balance = pd.DataFrame(rows)
display(month_balance)

In [ ]:
# Days by year from removal (removal year excluded). BACI needs controls in the same years as treated.
frame = pd.concat([FRAMES[("distance_30km", s, "baci_distance")] for s in DM["series"]])
grid = frame.pivot_table(index=["series", "role"], columns="years_from_removal", values="y",
                         aggfunc="size", fill_value=0)
fig, ax = plt.subplots(figsize=(10, 0.35 * len(grid) + 1))
sns.heatmap(np.log10(grid.where(grid > 0)), cmap="Greys", ax=ax, cbar_kws={"label": "log10 days"})
ax.set(title="Days by year from removal (30 km BACI rows)", xlabel="years from removal", ylabel="")
fig.tight_layout()

In [ ]:
# Measurement mix pre vs post, on the raw rows behind the 30 km BACI days: provider,
# censoring, assumed nutrient basis, turbidity unit. A big change between periods is a
# method change, not a removal effect.
key = ["AR_ID", "station_id", "series", "sample_date"]
days = pd.concat([FRAMES[("distance_30km", s, "baci_distance")][key] for s in DM["series"]]).drop_duplicates()
kept_keyed = kept.assign(sample_date=kept["sample_date"].astype(str))
days = days.assign(sample_date=days["sample_date"].astype(str))
raw = kept_keyed.merge(days, on=key, how="inner")
raw["role"] = raw["role_distance_30km"]
raw["nwis"] = raw["provider"].eq("NWIS")
raw["censored"] = raw["is_censored"].fillna(False).astype(bool)
raw["basis_assumed"] = raw["unit_basis_assumed"].fillna(False).astype(bool)

mix = (raw.groupby(["series", "role", "period"])[["nwis", "censored", "basis_assumed"]].mean()
       .mul(100).unstack("period"))
mix.columns = [f"{c}_{p}_%" for c, p in mix.columns]
for c in ("nwis", "censored", "basis_assumed"):
    mix[f"{c}_change_pts"] = mix[f"{c}_post_%"] - mix[f"{c}_pre_%"]
display(mix.round(1))

turb = raw[raw["parameter_std"].eq("turbidity")]
if len(turb):
    display(pd.crosstab([turb["role"], turb["period"]],
                        turb["result_unit_raw"].str.strip().str.upper(), normalize="index")
            .mul(100).round(1))

## 6. Outcome distributions and outliers

`y` is on the model scale (log10 for skewed parameters). Outliers: daily values more than 5 robust SDs
(MAD × 1.4826) from their own station's median. They aren't removed by the models; a handful at one dam
can move that dam's effect.

In [ ]:
fig, axes = facet_axes(len(DM["series"]))
for ax, series in zip(axes, DM["series"]):
    frame = FRAMES[("distance_30km", series, "before_after_distance")]
    base = FRAMES[("distance_30km", series, "baci_distance")]
    both = pd.concat([frame.assign(group="treated"), base[base["role"] == "control"].assign(group="control")])
    if both.empty:
        ax.set_title(f"{series}: no rows"); continue
    sns.boxplot(data=both, x="group", y="y", hue="period", hue_order=["pre", "post"],
                showfliers=False, ax=ax)
    ax.set(title=series, xlabel="", ylabel=y_label(series))
    ax.legend(fontsize=7)
fig.tight_layout()

In [ ]:
outliers = []
for series in DM["series"]:
    frame = FRAMES[("distance_30km", series, "before_after_distance")]
    if frame.empty:
        continue
    g = frame.groupby("station_id")["y"]
    mad = g.transform(lambda v: stats.median_abs_deviation(v, scale="normal"))
    z = (frame["y"] - g.transform("median")) / mad.replace(0, np.nan)
    flag = z.abs() > 5
    outliers.append({"series": series, "days": len(frame), "outlier_days": int(flag.sum()),
                     "outlier_%": 100 * flag.mean(),
                     "dams_with_outliers": frame.loc[flag, "AR_ID"].nunique(),
                     "top_dam": frame.loc[flag, "AR_ID"].value_counts().index[0] if flag.any() else None,
                     "post_share_of_outliers": frame.loc[flag, "post"].mean() if flag.any() else np.nan})
display(pd.DataFrame(outliers))
# post_share_of_outliers far from the overall post share (section 2) means outliers sit in one period.

## 7. A model-free BACI check, one value per dam

For each dam: remove each station's seasonal cycle (value minus that station's mean for the calendar
month, over both periods), average per station, then per role, and take after − before.
**BACI change** = treated change − control change. Each dam counts once, as in the models' dam-level
inference. If the model's estimate and this simple dam-level summary disagree in sign, look for the
dams driving the model (section 4) before trusting either.

In [ ]:
def dam_changes(frame):
    """One row per dam: deseasonalized after - before for each role, and their difference."""
    f = frame.copy()
    f["anomaly"] = f["y"] - f.groupby(["station_id", "month"])["y"].transform("mean")
    station = f.groupby(["AR_ID", "station_id", "role", "post"])["anomaly"].mean()
    role = station.groupby(["AR_ID", "role", "post"]).mean().unstack("post")
    change = (role[1] - role[0]).unstack("role")
    if "control" in change:
        change["baci"] = change["treated"] - change["control"]
    return change


def summary(values):
    v = values.dropna()
    if len(v) < 2:
        return pd.Series({"dams": len(v)})
    half = stats.t.ppf(0.975, len(v) - 1) * v.std() / np.sqrt(len(v))
    return pd.Series({"dams": len(v), "mean": v.mean(), "ci_low": v.mean() - half,
                      "ci_high": v.mean() + half, "median": v.median(),
                      "share_positive_%": 100 * (v > 0).mean()})


checks, CHANGES = [], {}
for run, model, column in [("mixed_10km", "baci", "baci"), ("mixed_10km", "before_after", "treated"),
                           ("distance_30km", "baci_distance", "baci"),
                           ("distance_30km", "before_after_distance", "treated")]:
    for series in RUNS[run]["settings"]["series"]:
        frame = FRAMES[(run, series, model)]
        if frame.empty:
            continue
        change = dam_changes(frame)
        CHANGES[(run, series, model)] = change
        if column in change:
            checks.append({"run": run, "model": model, "series": series, **summary(change[column])})
checks = pd.DataFrame(checks)
display(checks)
# Units are the model's (log10 for skewed parameters: 0.1 = about +26%).

In [ ]:
# Dam-level BACI changes, 30 km rows: each dot is one dam; the bar is the mean with its t interval.
series_with_baci = [s for s in DM["series"] if ("distance_30km", s, "baci_distance") in CHANGES
                    and "baci" in CHANGES[("distance_30km", s, "baci_distance")]]
fig, axes = facet_axes(len(series_with_baci), ncols=4, width=3.6, height=2.8)
for ax, series in zip(axes, series_with_baci):
    values = CHANGES[("distance_30km", series, "baci_distance")]["baci"].dropna()
    s = summary(values)
    ax.scatter(values, np.random.default_rng(0).uniform(-0.3, 0.3, len(values)), s=12,
               color="grey", alpha=0.7)
    if "mean" in s:
        ax.errorbar(s["mean"], 0, xerr=[[s["mean"] - s["ci_low"]], [s["ci_high"] - s["mean"]]],
                    fmt="o", color="tab:blue", capsize=4)
    ax.axvline(0, color="black", lw=0.8)
    ax.set(title=f"{series} ({len(values)} dams)", yticks=[], xlabel=f"BACI change, {y_label(series)}")
fig.tight_layout()

## 8. Distance and dilution (treated rows)

The distance models estimate how the effect changes with `ln(1 + km)`. That slope is only learned
where treated stations spread over distance, and it is learned **within** a dam only when the dam has
downstream stations at several distances; otherwise it compares different dams at different distances.
The drainage-area ratio (dilution proxy) also enters the effect, so if distance and ratio move together
the two slopes share information (high VIF = wide intervals for both).

In [ ]:
pairs = []
for series in DM["series"]:
    frame = FRAMES[("distance_30km", series, "before_after_distance")]
    if frame.empty:
        continue
    p = (frame.groupby(["AR_ID", "station_id"])
         .agg(km=("network_distance_km", "first"), dar=("drainage_area_ratio", "first"),
              days=("y", "size")).reset_index().assign(series=series))
    pairs.append(p)
pairs = pd.concat(pairs, ignore_index=True)
pairs["band"] = pd.cut(pairs["km"], BAND_EDGES, include_lowest=True)

rows = []
for series, p in pairs.groupby("series"):
    per_dam = p.groupby("AR_ID")["km"].agg(["size", lambda k: k.max() - k.min()])
    ld = np.log1p(p["km"]); ldar = np.log10(p["dar"])
    r = ld.corr(ldar)
    rows.append({"series": series, "pairs": len(p), "dams": p["AR_ID"].nunique(),
                 "median_km": p["km"].median(),
                 "dams_with_2plus_stations": int((per_dam["size"] >= 2).sum()),
                 "dams_with_5km_spread": int((per_dam.iloc[:, 1] >= 5).sum()),
                 "r_ld_ldar": r, "vif": 1 / (1 - r ** 2) if abs(r) < 1 else np.inf})
display(pd.DataFrame(rows).sort_values("dams", ascending=False))

display(pairs.pivot_table(index="series", columns="band", values="days", aggfunc="sum",
                          fill_value=0, observed=False))

In [ ]:
# Treated pair change vs distance (before/after, deseasonalized as in section 7), with a LOWESS
# line. FOCUS one series at a time.
from statsmodels.nonparametric.smoothers_lowess import lowess

FOCUS = ["water_temp", "dissolved_oxygen", "ph", "specific_conductance"]
fig, axes = facet_axes(len(FOCUS), ncols=4, width=4, height=3)
for ax, series in zip(axes, FOCUS):
    frame = FRAMES[("distance_30km", series, "before_after_distance")]
    if frame.empty:
        ax.set_title(f"{series}: no rows"); continue
    f = frame.assign(anomaly=frame["y"] - frame.groupby(["station_id", "month"])["y"].transform("mean"))
    station = f.groupby(["AR_ID", "station_id", "post"]).agg(a=("anomaly", "mean"),
                                                             km=("network_distance_km", "first"))
    change = (station["a"].unstack("post").pipe(lambda t: t[1] - t[0]).rename("change")
              .to_frame().join(station["km"].groupby(level=[0, 1]).first()).dropna())
    ax.scatter(change["km"], change["change"], s=10, color="grey", alpha=0.6)
    if len(change) > 10:
        line = lowess(change["change"], np.log1p(change["km"]), frac=0.6)
        ax.plot(np.expm1(line[:, 0]), line[:, 1], color="tab:blue")
    ax.axhline(0, color="black", lw=0.8)
    ax.set(xscale="symlog", title=series, xlabel="km downstream", ylabel=f"change, {y_label(series)}")
fig.tight_layout()

## 9. Reservoir size (size models)

`reservoir_volume_proxy` is a constructed index (height² ÷ slope × √drainage area), only meaningful as a
rank or on a log scale. The size models drop dams without it or without a dam drainage area (never
imputed). Size and river size are standardized over dams and both enter the effect, so their
correlation matters.

In [ ]:
dams = (DAILY["distance_30km"].groupby("AR_ID")
        [["reservoir_volume_proxy", "reservoir_polygon_area_km2", "reservoir_polygon_timing",
          "dam_drainage_area_km2", "backwater_km"]].first())
in_baci = set(pd.concat([FRAMES[("distance_30km", s, "baci_distance")]["AR_ID"] for s in DM["series"]]))
dams["in_any_baci"] = dams.index.isin(in_baci)
coverage = dams.groupby("in_any_baci").agg(
    dams=("backwater_km", "size"),
    proxy_known=("reservoir_volume_proxy", lambda v: v.gt(0).mean() * 100),
    polygon_known=("reservoir_polygon_area_km2", lambda v: v.gt(0).mean() * 100),
    backwater_known=("backwater_km", lambda v: v.gt(0).mean() * 100),
    dam_area_known=("dam_drainage_area_km2", lambda v: v.gt(0).mean() * 100))
display(coverage.round(1))

logs = np.log10(dams[["reservoir_volume_proxy", "reservoir_polygon_area_km2",
                      "dam_drainage_area_km2", "backwater_km"]].where(lambda t: t > 0))
display(logs.corr(method="spearman").round(2))
display(sizes[sizes["model"].str.endswith("_size")]
        .pivot_table(index="series", columns="model", values=["dams", "dams_before_modifier"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, column in zip(axes, ["reservoir_volume_proxy", "dam_drainage_area_km2", "backwater_km"]):
    ax.hist(logs[column].dropna(), bins=30, color="grey")
    ax.set(title=column, xlabel="log10")
axes[0].set_ylabel("dams (30 km run)")
fig.tight_layout()

## 10. Dam, watershed and climate characteristics

What the dams, stations and sampling days in the fitted models look like. Dams are split into three groups,
coloured as on the map: **BACI** (in at least one fitted BACI model), **before/after only**, and
**not modeled** (in `analysis_measurement` but in no fitted model of this run). Comparing the first two with
the third shows whether the model dams are typical of the removals we have data for.

- Dam level: height, age, year built and removed, original use, reservoir size, dam drainage area.
- Station level (model dam-station pairs, treated vs control): upstream area, baseline land cover,
  drainage-area ratio and distance.
- Day level (model days): air temperature and precipitation.

Skewed measures are shown on a log10 scale. Set `CHAR_RUN = "mixed_10km"` for the 10 km models.

In [ ]:
CHAR_RUN = "distance_30km"
s = RUNS[CHAR_RUN]["settings"]
CHAR_SPECS = {"distance_30km": {"baci": "baci_distance", "before_after": "before_after_distance"},
              "mixed_10km": {"baci": "baci", "before_after": "before_after"}}[CHAR_RUN]
GROUP_COLORS = {"BACI": "#c0392b", "before/after only": "#2980b9", "not modeled": "#999999"}
ROLE_COLORS = {"treated": "#c0392b", "control": "#2980b9"}
LOG_COLUMNS = {"dam_height_ft", "reservoir_volume_proxy", "reservoir_polygon_area_km2", "backwater_km",
               "dam_drainage_area_km2", "upstream_area_km2", "drainage_area_ratio"}

# Dams and days of every fitted model (a model with fewer than min_dams isn't fit).
baci_dams, ba_dams, day_frames = set(), set(), []
for series in s["series"]:
    for design, model in CHAR_SPECS.items():
        frame = FRAMES[(CHAR_RUN, series, model)]
        if frame["AR_ID"].nunique() < s["min_dams"]:
            continue
        (baci_dams if design == "baci" else ba_dams).update(frame["AR_ID"].astype(str))
        day_frames.append(frame[["AR_ID", "station_id", "sample_date", "role", "period", "month"]
                                + s["covariates"]])
# One row per dam, station and day (the same day appears once per measure).
model_days = pd.concat(day_frames).drop_duplicates(["AR_ID", "station_id", "sample_date"])
model_pairs = model_days[["AR_ID", "station_id", "role"]].drop_duplicates()

DAM_NUMERIC = [c for c in ["dam_height_ft", "dam_age_at_removal", "year_built", "removal_year",
                           "reservoir_volume_proxy", "reservoir_polygon_area_km2", "backwater_km",
                           "dam_drainage_area_km2"] if c in df.columns]
dam_table = df.assign(AR_ID=df["AR_ID"].astype(str)).groupby("AR_ID")[
    DAM_NUMERIC + [c for c in ["original_use", "state"] if c in df.columns]].first()
dam_table["group"] = np.select([dam_table.index.isin(baci_dams), dam_table.index.isin(ba_dams)],
                               ["BACI", "before/after only"], "not modeled")
for c in ("reservoir_volume_proxy", "reservoir_polygon_area_km2", "backwater_km", "dam_drainage_area_km2",
          "dam_height_ft"):
    if c in dam_table:
        dam_table[c] = dam_table[c].where(dam_table[c] > 0)   # log scale: non-positive = unknown
print(dam_table["group"].value_counts().to_string())


def describe(table, columns, by):
    """Per group: known values, share missing, 10th / 50th / 90th percentile."""
    out = {}
    for c in columns:
        g = table.groupby(by)[c]
        out[c] = pd.DataFrame({"known": g.count(), "missing_%": 100 * (1 - g.count() / g.size()),
                               "p10": g.quantile(0.1), "median": g.median(), "p90": g.quantile(0.9)})
    return pd.concat(out, names=["covariate", by])


def group_hists(table, columns, by, colors, ncols=4):
    """Overlaid density histograms per column, one outline per group, shared bins."""
    fig, axes = facet_axes(len(columns), ncols=ncols, width=3.8, height=2.8)
    for ax, c in zip(axes, columns):
        values = pd.to_numeric(table[c], errors="coerce")
        log = c in LOG_COLUMNS
        values = np.log10(values.where(values > 0)) if log else values
        known = values.dropna()
        if known.empty:
            ax.set_title(f"{c}: no values"); continue
        bins = np.histogram_bin_edges(known, bins=25)
        for group, color in colors.items():
            v = values[table[by] == group].dropna()
            if len(v):
                ax.hist(v, bins=bins, density=True, histtype="step", lw=1.6, color=color,
                        label=f"{group} ({len(v)})")
        ax.set(title=c, xlabel=f"log10 {c}" if log else c, yticks=[])
        ax.legend(fontsize=7)
    fig.tight_layout()

In [ ]:
# Dam characteristics by group.
display(describe(dam_table, DAM_NUMERIC, "group").unstack("group").round(2))
group_hists(dam_table, DAM_NUMERIC, "group", GROUP_COLORS)

In [ ]:
# Original use and decade of removal, % of each group's dams.
if "original_use" in dam_table:
    use = dam_table["original_use"].fillna("unknown").str.split(r"[;,]").str[0].str.strip()
    top = use.value_counts().index[:10]
    display(pd.crosstab(use.where(use.isin(top), "other"), dam_table["group"], normalize="columns")
            .mul(100).round(1))
decade = (dam_table["removal_year"] // 10 * 10).astype("Int64").astype(str) + "s"
display(pd.crosstab(decade, dam_table["group"], normalize="columns").mul(100).round(1))

In [ ]:
# Station-level characteristics of the model dam-station pairs. Land cover is the baseline year
# before removal (one value per pair); upstream area is null unless drainage_area_flag is 'ok'.
# For control (upstream) stations, drainage_area_ratio is below 1 and distance is upstream: the
# models use neither for controls (distance and dilution enter only downstream).
PAIR_NUMERIC = [c for c in ["upstream_area_km2", "drainage_area_ratio", "network_distance_km",
                            "pct_forest", "pct_developed", "pct_agriculture", "pct_wetlands",
                            "pct_water", "pct_other", "land_cover_year"] if c in df.columns]
pair_values = (df.assign(AR_ID=df["AR_ID"].astype(str))
               .groupby(["AR_ID", "station_id"])[PAIR_NUMERIC].first().reset_index())
pairs_table = model_pairs.assign(AR_ID=model_pairs["AR_ID"].astype(str)).merge(
    pair_values, on=["AR_ID", "station_id"], how="left")
print(pairs_table["role"].value_counts().to_string())
display(describe(pairs_table, PAIR_NUMERIC, "role").unstack("role").round(2))
group_hists(pairs_table, [c for c in PAIR_NUMERIC if c != "land_cover_year"], "role", ROLE_COLORS)

In [ ]:
# Land cover side by side: treated vs control stations (% of upstream area).
LC = [c for c in PAIR_NUMERIC if c.startswith("pct_")]
if LC:
    long = pairs_table.melt(id_vars=["role"], value_vars=LC, var_name="class", value_name="pct")
    long["class"] = long["class"].str.removeprefix("pct_")
    fig, ax = plt.subplots(figsize=(10, 3.5))
    sns.boxplot(data=long, x="class", y="pct", hue="role", palette=ROLE_COLORS, showfliers=False, ax=ax)
    ax.set(xlabel="", ylabel="% of upstream area", title="Baseline land cover of model stations")
    fig.tight_layout()

In [ ]:
# Climate on the model days, by role and period. air_temp_mean_c and precip_prev_month_mm are the
# model covariates; precip_mm (the sample month) is shown for context.
days = model_days.copy()
if "precip_mm" in df.columns:
    month_precip = (df.assign(AR_ID=df["AR_ID"].astype(str))
                    .drop_duplicates(["AR_ID", "station_id", "sample_date"])
                    [["AR_ID", "station_id", "sample_date", "precip_mm"]])
    days = days.assign(AR_ID=days["AR_ID"].astype(str)).merge(
        month_precip, on=["AR_ID", "station_id", "sample_date"], how="left")
CLIMATE = [c for c in ["air_temp_mean_c", "precip_mm", "precip_prev_month_mm"] if c in days.columns]
display(describe(days.assign(group=days["role"] + " " + days["period"]), CLIMATE, "group")
        .unstack("group").round(1))
fig, axes = plt.subplots(1, len(CLIMATE), figsize=(4.2 * len(CLIMATE), 3.2), squeeze=False)
for ax, c in zip(axes.flat, CLIMATE):
    sns.boxplot(data=days, x="role", y=c, hue="period", hue_order=["pre", "post"],
                showfliers=False, ax=ax)
    ax.set(title=c, xlabel="")
fig.tight_layout()

# Air temperature through the year: the seasonal spread the month effects absorb.
fig, ax = plt.subplots(figsize=(8, 3))
sns.boxplot(data=days, x="month", y="air_temp_mean_c", color="lightgrey", showfliers=False, ax=ax)
ax.set(title="Air temperature of model days by month", ylabel="°C")
fig.tight_layout()

In [ ]:
# Spearman correlations among dam and station covariates, treated model pairs.
treated = pairs_table[pairs_table["role"] == "treated"].merge(
    dam_table[DAM_NUMERIC], left_on="AR_ID", right_index=True, how="left")
cols = [c for c in DAM_NUMERIC + PAIR_NUMERIC if c != "land_cover_year" and treated[c].notna().sum() > 10]
corr = treated[cols].corr(method="spearman")
fig, ax = plt.subplots(figsize=(0.55 * len(cols) + 3, 0.5 * len(cols) + 2))
sns.heatmap(corr, cmap="RdBu_r", vmin=-1, vmax=1, annot=True, fmt=".1f", annot_kws={"size": 7}, ax=ax)
ax.set(title=f"Spearman correlation, {len(treated)} treated model pairs")
fig.tight_layout()
# Strong pairs (|rho| > 0.7) carry the same information; don't read them as separate effects.
strong = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack()
display(strong[strong.abs() > 0.7].sort_values().rename("rho").to_frame())

## 11. Covariate shifts before vs after

The models adjust for that month's upstream air temperature and the previous month's rainfall
(standardized within each fit). A covariate that differs a lot between periods is doing real work;
if it differs differently for treated and control stations, BACI leans on it.

In [ ]:
cov = []
for series in DM["series"]:
    frame = FRAMES[("distance_30km", series, "baci_distance")]
    for c in DM["covariates"]:
        for role, part in frame.groupby("role"):
            pre, post = part.loc[part["post"] == 0, c], part.loc[part["post"] == 1, c]
            sd = part[c].std()
            cov.append({"series": series, "covariate": c, "role": role,
                        "pre_mean": pre.mean(), "post_mean": post.mean(),
                        "shift_in_sd": (post.mean() - pre.mean()) / sd if sd else np.nan})
cov = pd.DataFrame(cov)
display(cov.pivot_table(index="series", columns=["covariate", "role"], values="shift_in_sd").round(2))
display(sizes[sizes["run"] == "distance_30km"]
        .pivot_table(index="series", columns="model", values="rows_dropped_missing_covariates"))

## 12. Checklist

One line per thing worth checking before reading a series' model results. Thresholds are rules of thumb
for this notebook only.

In [ ]:
notes = []
for _, r in sizes[(sizes["run"] == "distance_30km") & sizes["model"].isin(["baci_distance", "before_after_distance"])].iterrows():
    if r["fit"] != "fit":
        notes.append((r["series"], r["model"], f"{r['fit']}: {r['dams']} dams"))
for (run, series, model), r in conc.iterrows():
    if r["top_dam_share_%"] > 30:
        notes.append((series, model, f"{r['top_dam']} holds {r['top_dam_share_%']:.0f}% of days"))
for _, r in month_balance.iterrows():
    if r["month_tvd"] > 0.2:
        notes.append((r["series"], "baci_distance", f"{r['role']} month mix shifts (TVD {r['month_tvd']:.2f})"))
for (series, role), r in mix.iterrows():
    for c in ("nwis", "censored", "basis_assumed"):
        if abs(r.get(f"{c}_change_pts", 0)) > 15:
            notes.append((series, "baci_distance",
                          f"{role} {c} share changes {r[f'{c}_change_pts']:+.0f} pts pre to post"))
for _, r in checks[checks["run"].eq("distance_30km")].iterrows():
    if r["dams"] >= 5 and (r["ci_low"] > 0 or r["ci_high"] < 0):
        notes.append((r["series"], r["model"], f"dam-level mean change {r['mean']:+.3g} excludes 0"))
checklist = pd.DataFrame(notes, columns=["series", "model", "note"]).sort_values(["series", "model"])
display(checklist.reset_index(drop=True))